# Algoritmos de cadenas · Tries y arrays de sufijos

Estructuras para preguntar muchas cosas a un mismo texto o a un mismo diccionario. Primero, el **trie** con contadores (búsqueda, prefijos, borrado y autocompletado por frecuencia) y su versión comprimida, el **radix tree**. Después, el **array de sufijos**: construcción ingenua y por duplicación de prefijos en $O(n \log^2 n)$, búsqueda de patrones con bisección, el array **LCP** con el algoritmo de Kasai en $O(n)$ y sus aplicaciones: subcadena repetida más larga, número de subcadenas distintas, subcadena común más larga de dos cadenas y la $k$-ésima subcadena en orden. Cada resultado se compara con una solución de fuerza bruta. Ideas de [cp-algorithms](https://cp-algorithms.com/string/suffix-array.html) y de la [guía de USACO](https://usaco.guide/adv/suffix-array).

In [1]:
import heapq
import random
from bisect import bisect_left, bisect_right
from collections import Counter

## 1. Trie con contadores

Crear un trie en el que cada nodo guarde cuántas palabras terminan en él y cuántas pasan por él. Implementar `insertar`, `buscar` (¿está la palabra?) y `contar_prefijo` (¿cuántas palabras empiezan así?), cargarlo con municipios canarios y comprobar las respuestas contra una búsqueda lineal en la lista.

In [2]:
class NodoTrie:
    __slots__ = ("hijos", "fin", "pasan")

    def __init__(self):
        self.hijos = {}
        self.fin = 0
        self.pasan = 0


class Trie:
    def __init__(self, palabras=()):
        self.raiz = NodoTrie()
        for p in palabras:
            self.insertar(p)

    def insertar(self, palabra, veces=1):
        nodo = self.raiz
        nodo.pasan += veces
        for c in palabra:
            nodo = nodo.hijos.setdefault(c, NodoTrie())
            nodo.pasan += veces
        nodo.fin += veces

    def _nodo(self, prefijo):
        nodo = self.raiz
        for c in prefijo:
            nodo = nodo.hijos.get(c)
            if nodo is None:
                return None
        return nodo

    def buscar(self, palabra):
        nodo = self._nodo(palabra)
        return nodo is not None and nodo.fin > 0

    def contar_prefijo(self, prefijo):
        nodo = self._nodo(prefijo)
        return nodo.pasan if nodo else 0

    def num_nodos(self):
        pila, total = [self.raiz], 0
        while pila:
            nodo = pila.pop()
            total += 1
            pila.extend(nodo.hijos.values())
        return total


municipios = ["tejeda", "telde", "teror", "tegueste", "teguise", "tias", "tinajo", "tijarafe",
              "tazacorte", "tuineje", "garachico", "galdar", "guia", "guimar", "garafia", "agaete",
              "agulo", "arico", "arona", "arucas", "artenara", "adeje", "alajero", "antigua"]
trie = Trie(municipios)
for prefijo in ["te", "teg", "ti", "ga", "gu", "ar", "a", "z", ""]:
    esperado = sum(m.startswith(prefijo) for m in municipios)
    print(f"{prefijo!r:6} -> {trie.contar_prefijo(prefijo)}")
    assert trie.contar_prefijo(prefijo) == esperado
for palabra in ["teror", "tero", "terore", "arona", "aro", "agulo"]:
    assert trie.buscar(palabra) == (palabra in municipios)
print("Nodos del trie:", trie.num_nodos(), "para", sum(map(len, municipios)), "letras")

'te'   -> 5
'teg'  -> 2
'ti'   -> 3
'ga'   -> 3
'gu'   -> 2
'ar'   -> 4
'a'    -> 9
'z'    -> 0
''     -> 24
Nodos del trie: 113 para 151 letras


## 2. Borrar palabras y podar nodos

Añadir al trie un método `borrar(palabra)` que devuelva `False` si la palabra no está y, si está, descuente los contadores y elimine los nodos por los que ya no pasa ninguna palabra. Borrar la mitad de los municipios y comprobar que el trie resultante tiene los mismos nodos y respuestas que uno construido desde cero con los que quedan.

In [3]:
def borrar(self, palabra):
    if not self.buscar(palabra):
        return False
    nodo = self.raiz
    nodo.pasan -= 1
    for c in palabra:
        hijo = nodo.hijos[c]
        hijo.pasan -= 1
        if hijo.pasan == 0:
            del nodo.hijos[c]  # poda toda la rama de una vez
            return True
        nodo = hijo
    nodo.fin -= 1
    return True


Trie.borrar = borrar

random.seed(7)
quitados = random.sample(municipios, len(municipios) // 2)
for m in quitados:
    assert trie.borrar(m)
print("Borrar uno que ya no está:", trie.borrar(quitados[0]))
print("Borrar un prefijo que no es palabra:", trie.borrar("te"))

quedan = [m for m in municipios if m not in quitados]
nuevo = Trie(quedan)
print(f"Nodos tras borrar: {trie.num_nodos()}; trie nuevo con {len(quedan)} palabras: {nuevo.num_nodos()}")
assert trie.num_nodos() == nuevo.num_nodos()
for prefijo in ["", "t", "te", "g", "a", "ar"]:
    assert trie.contar_prefijo(prefijo) == nuevo.contar_prefijo(prefijo)
assert all(trie.buscar(m) == (m in quedan) for m in municipios)

Borrar uno que ya no está: False
Borrar un prefijo que no es palabra: False
Nodos tras borrar: 60; trie nuevo con 12 palabras: 60


## 3. Autocompletado por frecuencia

Un buscador de una web de turismo guarda cuántas veces se ha buscado cada término. Cargar el registro en un trie (con `insertar(palabra, veces)`) y escribir `sugerir(prefijo, k)`, que devuelva las `k` palabras con ese prefijo más buscadas, desempatando por orden alfabético. Comparar con ordenar todo el registro.

In [4]:
def sugerir(self, prefijo, k=3):
    nodo = self._nodo(prefijo)
    if nodo is None:
        return []
    candidatas, pila = [], [(nodo, prefijo)]
    while pila:
        nodo, palabra = pila.pop()
        if nodo.fin:
            candidatas.append((-nodo.fin, palabra))
        pila.extend((hijo, palabra + c) for c, hijo in nodo.hijos.items())
    return [(p, -f) for f, p in heapq.nsmallest(k, candidatas)]


Trie.sugerir = sugerir

terminos = ["playa", "playa de las teresitas", "playa del ingles", "playa de papagayo", "plaza", "planetario",
            "teide", "teide teleferico", "teleferico", "tenerife", "teror", "termas", "papas arrugadas", "parque rural"]
rng = random.Random(2024)
registro = Counter({t: rng.randint(1, 60) for t in terminos})
registro["playa del ingles"] = registro["playa de papagayo"]  # fuerza un empate

buscador = Trie()
for termino, veces in registro.items():
    buscador.insertar(termino, veces)

for prefijo in ["pla", "playa d", "te", "tel", "pa", "x"]:
    sugerencias = buscador.sugerir(prefijo, 3)
    print(f"{prefijo!r:10} {sugerencias}")
    fuerza = sorted(((t, f) for t, f in registro.items() if t.startswith(prefijo)), key=lambda x: (-x[1], x[0]))[:3]
    assert sugerencias == fuerza

'pla'      [('playa de papagayo', 38), ('playa del ingles', 38), ('playa', 31)]
'playa d'  [('playa de papagayo', 38), ('playa del ingles', 38), ('playa de las teresitas', 12)]
'te'       [('teide', 57), ('tenerife', 49), ('termas', 49)]
'tel'      [('teleferico', 27)]
'pa'       [('parque rural', 35), ('papas arrugadas', 17)]
'x'        []


## 4. Trie comprimido (radix tree)

En un trie, las cadenas de nodos con un solo hijo gastan memoria. Crear un *radix tree* en el que cada arista lleva una etiqueta de varias letras: al insertar, si la palabra comparte solo una parte de una etiqueta, la arista se parte en dos. Implementar `insertar`, `buscar`, `palabras` y un dibujo con sangría, y comprobarlo con los municipios.

In [5]:
def prefijo_comun(a, b):
    k = 0
    while k < min(len(a), len(b)) and a[k] == b[k]:
        k += 1
    return k


class NodoRadix:
    __slots__ = ("hijos", "fin")

    def __init__(self, fin=False):
        self.hijos = {}  # primera letra -> (etiqueta, nodo)
        self.fin = fin


class Radix:
    def __init__(self, palabras=()):
        self.raiz = NodoRadix()
        for p in palabras:
            self.insertar(p)

    def insertar(self, palabra):
        nodo, resto = self.raiz, palabra
        while resto:
            if resto[0] not in nodo.hijos:
                nodo.hijos[resto[0]] = (resto, NodoRadix(fin=True))
                return
            etiqueta, hijo = nodo.hijos[resto[0]]
            k = prefijo_comun(etiqueta, resto)
            if k < len(etiqueta):
                medio = NodoRadix()
                medio.hijos[etiqueta[k]] = (etiqueta[k:], hijo)
                nodo.hijos[resto[0]] = (etiqueta[:k], medio)
                hijo = medio
            nodo, resto = hijo, resto[k:]
        nodo.fin = True

    def buscar(self, palabra):
        nodo, resto = self.raiz, palabra
        while resto:
            if resto[0] not in nodo.hijos:
                return False
            etiqueta, hijo = nodo.hijos[resto[0]]
            if not resto.startswith(etiqueta):
                return False
            nodo, resto = hijo, resto[len(etiqueta):]
        return nodo.fin

    def palabras(self, nodo=None, prefijo=""):
        nodo = nodo or self.raiz
        if nodo.fin:
            yield prefijo
        for c in sorted(nodo.hijos):
            etiqueta, hijo = nodo.hijos[c]
            yield from self.palabras(hijo, prefijo + etiqueta)

    def dibujar(self, nodo=None, nivel=0):
        nodo = nodo or self.raiz
        for c in sorted(nodo.hijos):
            etiqueta, hijo = nodo.hijos[c]
            print("  " * nivel + etiqueta + (" *" if hijo.fin else ""))
            self.dibujar(hijo, nivel + 1)


radix = Radix(m for m in municipios if m.startswith("t"))
radix.dibujar()
assert list(radix.palabras()) == sorted(m for m in municipios if m.startswith("t"))
for palabra in ["teguise", "tegu", "te", "tias", "tiasa", "tinajo", "z"]:
    assert radix.buscar(palabra) == (palabra in municipios)

t
  azacorte *
  e
    gu
      este *
      ise *
    jeda *
    lde *
    ror *
  i
    as *
    jarafe *
    najo *
  uineje *


## 5. Nodos de un trie frente a un radix tree

Generar 2000 palabras aleatorias de entre 3 y 12 letras sobre un alfabeto de 4 letras, guardarlas en un trie y en un radix tree y comparar el número de nodos. Comprobar la propiedad que garantiza el ahorro: en el radix tree, todo nodo que no es la raíz ni final de palabra tiene al menos dos hijos, así que hay menos de dos nodos por palabra.

In [6]:
def nodos_radix(nodo):
    return 1 + sum(nodos_radix(h) for _, h in nodo.hijos.values())


def comprobar_ramas(nodo, es_raiz=True):
    assert es_raiz or nodo.fin or len(nodo.hijos) >= 2
    for _, hijo in nodo.hijos.values():
        comprobar_ramas(hijo, False)


rng = random.Random(5)
palabras = {"".join(rng.choices("acgt", k=rng.randint(3, 12))) for _ in range(2000)}
t, r = Trie(palabras), Radix(palabras)
print(f"Palabras: {len(palabras)}; letras: {sum(map(len, palabras))}")
print(f"Nodos del trie: {t.num_nodos()}; nodos del radix: {nodos_radix(r.raiz)}")
comprobar_ramas(r.raiz)
assert nodos_radix(r.raiz) < 2 * len(palabras) + 1
assert set(r.palabras()) == palabras

Palabras: 1778; letras: 14362
Nodos del trie: 6586; nodos del radix: 2357


## 6. Array de sufijos ingenuo

El array de sufijos de una cadena es la lista de posiciones iniciales de sus sufijos ordenados alfabéticamente. Construirlo ordenando los sufijos completos (coste $O(n^2 \log n)$ en el peor caso) y mostrar la tabla de `"barrancobarra"`.

In [7]:
def sufijos_ingenuo(s):
    return sorted(range(len(s)), key=lambda i: s[i:])


texto = "barrancobarra"
sa = sufijos_ingenuo(texto)
for pos, i in enumerate(sa):
    print(f"{pos:2} {i:3}  {texto[i:]}")
assert [texto[i:] for i in sa] == sorted(texto[i:] for i in range(len(texto)))

 0  12  a
 1   4  ancobarra
 2   9  arra
 3   1  arrancobarra
 4   8  barra
 5   0  barrancobarra
 6   6  cobarra
 7   5  ncobarra
 8   7  obarra
 9  11  ra
10   3  rancobarra
11  10  rra
12   2  rrancobarra


## 7. Duplicación de prefijos en $O(n \log^2 n)$

Construir el array de sufijos ordenando por los primeros $k$ caracteres con $k = 1, 2, 4, \ldots$: el rango de un sufijo para $2k$ es el par (rango del sufijo en $k$, rango del sufijo que empieza $k$ posiciones después), y cada ronda es una ordenación. Comprobarlo con la versión ingenua en 300 cadenas aleatorias de varios alfabetos.

In [8]:
def sufijos_duplicando(s):
    n = len(s)
    rango = [ord(c) for c in s]
    sa = list(range(n))
    k = 1
    while True:
        clave = lambda i: (rango[i], rango[i + k] if i + k < n else -1)
        sa.sort(key=clave)
        nuevo = [0] * n
        for a, b in zip(sa, sa[1:]):
            nuevo[b] = nuevo[a] + (clave(a) != clave(b))
        rango = nuevo
        if n == 0 or rango[sa[-1]] == n - 1:  # todos los rangos distintos
            return sa
        k *= 2


print(sufijos_duplicando(texto))
rng = random.Random(11)
for _ in range(300):
    s = "".join(rng.choices(rng.choice(["a", "ab", "acgt", "abcdefghij"]), k=rng.randint(0, 60)))
    assert sufijos_duplicando(s) == sufijos_ingenuo(s), s
print("300 cadenas aleatorias correctas")

[12, 4, 9, 1, 8, 0, 6, 5, 7, 11, 3, 10, 2]
300 cadenas aleatorias correctas


## 8. Buscar un patrón con el array de sufijos

Todas las apariciones de un patrón son sufijos que empiezan por él, y en el array de sufijos están seguidos. Localizarlos con dos búsquedas binarias (`bisect` con `key`), en $O(m \log n)$, y devolver las posiciones ordenadas. Comparar con `str.startswith` en todas las posiciones.

In [9]:
def apariciones(s, sa, patron):
    m = len(patron)
    clave = lambda i: s[i:i + m]
    izq = bisect_left(sa, patron, key=clave)
    der = bisect_right(sa, patron, key=clave)
    return sorted(sa[izq:der])


rng = random.Random(3)
adn = "".join(rng.choices("acgt", k=5000))
sa_adn = sufijos_duplicando(adn)
for patron in ["gatta", "acgt", "tttt", "c", adn[1000:1012], "gattaca"]:
    pos = apariciones(adn, sa_adn, patron)
    print(f"{patron:10} {len(pos):4} apariciones; primeras: {pos[:5]}")
    assert pos == [i for i in range(len(adn)) if adn.startswith(patron, i)]

gatta         4 apariciones; primeras: [164, 433, 3953, 4743]
acgt         20 apariciones; primeras: [157, 265, 326, 501, 653]
tttt         13 apariciones; primeras: [229, 1024, 1541, 1615, 1811]
c          1317 apariciones; primeras: [2, 8, 11, 13, 24]
aatctaatttgg    1 apariciones; primeras: [1000]
gattaca       0 apariciones; primeras: []


## 9. Array LCP con Kasai

El array LCP guarda, para cada par de sufijos vecinos en el array de sufijos, la longitud de su prefijo común. Calcularlo en $O(n)$ con el algoritmo de Kasai, que recorre los sufijos en orden de posición y aprovecha que el LCP baja como mucho 1 de un sufijo al siguiente. Comparar con el cálculo directo.

In [10]:
def kasai(s, sa):
    n = len(s)
    rango = [0] * n
    for pos, i in enumerate(sa):
        rango[i] = pos
    lcp, h = [0] * n, 0  # lcp[p]: prefijo común de sa[p-1] y sa[p]
    for i in range(n):
        if rango[i] > 0:
            j = sa[rango[i] - 1]
            while i + h < n and j + h < n and s[i + h] == s[j + h]:
                h += 1
            lcp[rango[i]] = h
            h = max(h - 1, 0)
        else:
            h = 0
    return lcp


def lcp_directo(s, sa):
    return [0] + [prefijo_comun(s[a:], s[b:]) for a, b in zip(sa, sa[1:])]


lcp = kasai(texto, sa)
for pos, i in enumerate(sa):
    print(f"{lcp[pos]:2}  {texto[i:]}")
rng = random.Random(13)
for _ in range(300):
    s = "".join(rng.choices(rng.choice(["a", "ab", "acgt"]), k=rng.randint(1, 60)))
    sa_s = sufijos_duplicando(s)
    assert kasai(s, sa_s) == lcp_directo(s, sa_s)

 0  a
 1  ancobarra
 1  arra
 4  arrancobarra
 0  barra
 5  barrancobarra
 0  cobarra
 0  ncobarra
 0  obarra
 0  ra
 2  rancobarra
 1  rra
 3  rrancobarra


## 10. Subcadena repetida más larga

La subcadena que aparece al menos dos veces y es más larga es el prefijo común de dos sufijos vecinos con el mayor LCP. Escribirla con el array de sufijos y Kasai (sin hashes, a diferencia de la versión de Rabin-Karp) y comprobar su longitud contra la fuerza bruta en cadenas aleatorias.

In [11]:
def repetida_mas_larga(s):
    sa = sufijos_duplicando(s)
    lcp = kasai(s, sa)
    if len(s) < 2:
        return ""
    pos = max(range(len(s)), key=lcp.__getitem__)
    return s[sa[pos]:sa[pos] + lcp[pos]]


def repetida_fuerza_bruta(s):
    for largo in range(len(s) - 1, 0, -1):
        vistas = Counter(s[i:i + largo] for i in range(len(s) - largo + 1))
        if max(vistas.values()) >= 2:
            return largo
    return 0


for s in ["barrancobarra", "papaparapapa", "abcdef", "aaaaa"]:
    r = repetida_mas_larga(s)
    print(f"{s:14} -> {r!r}")
    assert len(r) == repetida_fuerza_bruta(s)
rng = random.Random(17)
for _ in range(200):
    s = "".join(rng.choices("ab", k=rng.randint(1, 40)))
    r = repetida_mas_larga(s)
    assert len(r) == repetida_fuerza_bruta(s)
    assert not r or s.find(r, s.find(r) + 1) >= 0

barrancobarra  -> 'barra'
papaparapapa   -> 'apapa'
abcdef         -> ''
aaaaa          -> 'aaaa'


## 11. Número de subcadenas distintas

Cada sufijo aporta tantos prefijos como su longitud, pero los primeros `lcp` ya los aportó su vecino anterior. Calcular el número de subcadenas distintas (no vacías) como $\frac{n(n+1)}{2} - \sum \text{lcp}$ y comprobarlo con un conjunto de todas las subcadenas.

In [12]:
def subcadenas_distintas(s):
    n = len(s)
    return n * (n + 1) // 2 - sum(kasai(s, sufijos_duplicando(s)))


def subcadenas_conjunto(s):
    return len({s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)})


for s in ["barrancobarra", "aaaa", "abcd", "tenerife"]:
    print(f"{s:14} {subcadenas_distintas(s):4}")
    assert subcadenas_distintas(s) == subcadenas_conjunto(s)
rng = random.Random(19)
assert all(subcadenas_distintas(s) == subcadenas_conjunto(s)
           for s in ("".join(rng.choices("abc", k=rng.randint(1, 50))) for _ in range(200)))

barrancobarra    74
aaaa              4
abcd             10
tenerife         34


## 12. Subcadena común más larga de dos cadenas

Unir las dos cadenas con un separador que no aparece en ninguna, construir el array de sufijos y el LCP de la unión y quedarse con el mayor LCP entre dos sufijos vecinos que empiecen en cadenas distintas. Comparar con la programación dinámica $O(nm)$.

In [13]:
def comun_mas_larga(a, b):
    s = a + "\x01" + b
    sa = sufijos_duplicando(s)
    lcp = kasai(s, sa)
    mejor, inicio = 0, 0
    for p in range(1, len(s)):
        if (sa[p] < len(a)) != (sa[p - 1] < len(a)) and lcp[p] > mejor:
            mejor, inicio = lcp[p], sa[p]
    return s[inicio:inicio + mejor]


def comun_dp(a, b):
    mejor, fila = 0, [0] * (len(b) + 1)
    for x in a:
        nueva = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            if x == y:
                nueva[j] = fila[j - 1] + 1
                mejor = max(mejor, nueva[j])
        fila = nueva
    return mejor


pares = [("guanchesdetenerife", "tenerifeguanche"), ("lapalmabonita", "palmeral"), ("abc", "xyz")]
for a, b in pares:
    c = comun_mas_larga(a, b)
    print(f"{a} / {b} -> {c!r}")
    assert len(c) == comun_dp(a, b) and c in a and c in b
rng = random.Random(23)
for _ in range(200):
    a = "".join(rng.choices("acgt", k=rng.randint(1, 40)))
    b = "".join(rng.choices("acgt", k=rng.randint(1, 40)))
    c = comun_mas_larga(a, b)
    assert len(c) == comun_dp(a, b) and c in a and c in b

guanchesdetenerife / tenerifeguanche -> 'tenerife'
lapalmabonita / palmeral -> 'palm'
abc / xyz -> ''


## 13. La $k$-ésima subcadena en orden alfabético

Recorriendo el array de sufijos en orden, el sufijo `sa[p]` aporta en orden alfabético las subcadenas nuevas de longitudes `lcp[p] + 1` hasta su longitud. Usarlo para obtener la $k$-ésima subcadena distinta (empezando en 1) sin generar todas, y comprobarlo con la lista ordenada del conjunto de subcadenas.

In [14]:
def kesima_subcadena(s, k):
    sa = sufijos_duplicando(s)
    lcp = kasai(s, sa)
    for p, i in enumerate(sa):
        nuevas = len(s) - i - lcp[p]
        if k <= nuevas:
            return s[i:i + lcp[p] + k]
        k -= nuevas
    raise IndexError("no hay tantas subcadenas distintas")


s = "tagoror"
todas = sorted({s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)})
print([kesima_subcadena(s, k) for k in range(1, 11)])
assert [kesima_subcadena(s, k) for k in range(1, len(todas) + 1)] == todas
try:
    kesima_subcadena(s, len(todas) + 1)
except IndexError as e:
    print("k demasiado grande:", e)
rng = random.Random(29)
for _ in range(100):
    s = "".join(rng.choices("abc", k=rng.randint(1, 25)))
    todas = sorted({s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)})
    assert [kesima_subcadena(s, k) for k in range(1, len(todas) + 1)] == todas

['a', 'ag', 'ago', 'agor', 'agoro', 'agoror', 'g', 'go', 'gor', 'goro']
k demasiado grande: no hay tantas subcadenas distintas
